In [4]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [22]:
def llm(prompt):
    response = openai_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": "hello, are you working?"}]
    )
    return response.choices[0].message.content

llm("Hey, what's up?")


"Hello. Yes, I'm working and ready to assist you. How can I help you today?"

In [23]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [6]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [7]:
courses_raw

[{'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 471},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 253},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 118}]

In [8]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1380

In [9]:
documents[10]

{'id': '2b5ff70c77',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Do I need to enroll in the course before submitting homework?',
 'answer': 'No enrollment is required to submit homework. Just log into the homework form when it opens. The Airtable registration you may see is only for announcements; actual submissions are made on the course platform forms and via your GitHub as specified in the homework guidelines.'}

In [10]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"], # the section where the we are going search for
    keyword_fields=["course"] # this field have to math exactly (it is like filtering the data to one course so we can filter the data we want to search for)
)

index.fit(documents)

In [11]:
question= 'Do I need to enroll in the course before submitting homework?'

search_resluts = index.search(question,
                              boost_dict={'question':2.0 ,'section':0.5 }, # weghts : how important is each field . so if you find the results in the question field it is going to be 2 times more imprtant 
                              filter_dict={'course':"llm-zoomcamp"})

In [12]:
print(search_resluts)

[{'id': '5e4ca64cb5', 'course': 'llm-zoomcamp', 'section': 'Module 1 Homework', 'question': 'Do homework answers need to match the options exactly?', 'answer': 'Not always. If your numeric answer is close to one of the options, choose the closest option.\n\nSmall differences can come from:\n\n- Slightly different filtering.\n- Different dataset versions.\n- Floating-point or rounding differences.\n- Different model/provider behavior.\n\nIf your answer is far from every option, re-check the question, the dataset version, and the GitHub homework instructions.\n\nSee the general homework guidance: [homework logistics](https://datatalks.club/docs/courses/zoomcamp-logistics/homework/).'}, {'id': '53f15299b6', 'course': 'llm-zoomcamp', 'section': 'Module 1 Homework', 'question': 'Where can I find the homework questions?', 'answer': 'Homework links are available in the course GitHub repo and in the course management platform.\n\nFor the 2026 Module 1 homework, use:\n\n- [Module 1 cohort mater

In [13]:
def search(question, course='llm-zoomcamp'):
    boost_dict = {'question': 2.0, 'section': 0.5}
    filter_dict = {'course': course}

    return index.search(
    question,
    boost_dict=boost_dict,
    filter_dict=filter_dict,
    num_results=5
    )

In [14]:
search_results=search(question)
print(search_results)

[{'id': '5e4ca64cb5', 'course': 'llm-zoomcamp', 'section': 'Module 1 Homework', 'question': 'Do homework answers need to match the options exactly?', 'answer': 'Not always. If your numeric answer is close to one of the options, choose the closest option.\n\nSmall differences can come from:\n\n- Slightly different filtering.\n- Different dataset versions.\n- Floating-point or rounding differences.\n- Different model/provider behavior.\n\nIf your answer is far from every option, re-check the question, the dataset version, and the GitHub homework instructions.\n\nSee the general homework guidance: [homework logistics](https://datatalks.club/docs/courses/zoomcamp-logistics/homework/).'}, {'id': '53f15299b6', 'course': 'llm-zoomcamp', 'section': 'Module 1 Homework', 'question': 'Where can I find the homework questions?', 'answer': 'Homework links are available in the course GitHub repo and in the course management platform.\n\nFor the 2026 Module 1 homework, use:\n\n- [Module 1 cohort mater

In [15]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [16]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [17]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [18]:
context =build_context(search_results)
print(context)

Module 1 Homework
Q: Do homework answers need to match the options exactly?
A: Not always. If your numeric answer is close to one of the options, choose the closest option.

Small differences can come from:

- Slightly different filtering.
- Different dataset versions.
- Floating-point or rounding differences.
- Different model/provider behavior.

If your answer is far from every option, re-check the question, the dataset version, and the GitHub homework instructions.

See the general homework guidance: [homework logistics](https://datatalks.club/docs/courses/zoomcamp-logistics/homework/).

Module 1 Homework
Q: Where can I find the homework questions?
A: Homework links are available in the course GitHub repo and in the course management platform.

For the 2026 Module 1 homework, use:

- [Module 1 cohort materials](https://github.com/DataTalksClub/llm-zoomcamp/tree/main/cohorts/2026/01-agentic-rag)
- [Module 1 homework instructions](https://github.com/DataTalksClub/llm-zoomcamp/blob/mai

In [19]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [20]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
Do I need to enroll in the course before submitting homework?

Context:
Module 1 Homework
Q: Do homework answers need to match the options exactly?
A: Not always. If your numeric answer is close to one of the options, choose the closest option.

Small differences can come from:

- Slightly different filtering.
- Different dataset versions.
- Floating-point or rounding differences.
- Different model/provider behavior.

If your answer is far from every option, re-check the question, the dataset version, and the GitHub homework instructions.

See the general homework guidance: [homework logistics](https://datatalks.club/docs/courses/zoomcamp-logistics/homework/).

Module 1 Homework
Q: Where can I find the homework questions?
A: Homework links are available in the course GitHub repo and in the course management platform.

For the 2026 Module 1 homework, use:

- [Module 1 cohort materials](https://github.com/DataTalksClub/llm-zoomcamp/tree/main/cohorts/2026/01-agentic-rag)
- [Modu

In [ ]:
message_history = [
    {"role": "system", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]


response = openai_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=message_history
    )
response.choices[0].message.content


"I don't know. The context doesn't mention anything about enrolling in the course before submitting homework. It only discusses how to find homework questions, what to do if homework questions feel unclear, and how to submit homework for specific modules, but it doesn't address the enrollment requirement."

In [36]:
response.usage


CompletionUsage(completion_tokens=58, prompt_tokens=954, total_tokens=1012, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.052384812, prompt_time=0.095930476, completion_time=0.203615454, total_time=0.29954593)

In [38]:
def llm(instructions, user_prompt, model="llama-3.3-70b-versatile"):
    message_history = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]
    
    
    response = openai_client.chat.completions.create(
            model=model,
            messages=message_history
        )
    
    

    return response.choices[0].message.content

In [39]:
def rag(query,  model="llama-3.3-70b-versatile"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [40]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while the course is still accepting submissions. You can start with the provided resources and follow the weekly workflow as described. Keep in mind that to get a certificate, you need to finish the course with a "live" cohort and complete the required peer reviews.
